# Modeling pipeline analyses

End-to-end notebook for the **vocal-modeling pipelines** in `usv_playpen.modeling`. Run sections in order — earlier sections produce the artifacts that later sections consume.

**Workflow at a glance**

1. **Imports** — load every module the notebook uses.
2. **Extract modeling input data** — convert per-session loader output into the per-pipeline modeling-input pickle and run the predictor-diagnostics audits (`_collinearity.pkl` + `_timescales.pkl`).
3. **Predictor diagnostics plots** — visualise the audit artifacts before committing to model fitting.
4. **Model selection** — forward-stepwise feature selection, kicked off from the local node or the SLURM dispatcher.
5. **Consolidate per-feature / per-step pickles** — merge SLURM-array outputs into a single artifact per analysis.
6. **Univariate-model visualisations** — feature ranking, single-feature filters, raw feature differences.
7. **Model-selection visualisations** — performance curves over forward-stepwise iterations.
8. **Multinomial-specific visualisations** — class-resolved performance, selection trajectory, final filters.
9. **CNN deep-learning pipeline** — non-linear baseline for the continuous manifold-position model.
10. **Vocalisation embedding-space view** — UMAP grid annotated with cluster categories.

**Settings file**: every pipeline reads `_parameter_settings/modeling_settings.json` via `modeling_settings_dict=None` (loads the project default). To run a one-off override, pass an explicit dict.


In [ ]:
# Imports

import pathlib

from usv_playpen.os_utils import configure_path
from usv_playpen.plot_style import apply_plot_style

from usv_playpen.modeling.consolidate_univariate_results import consolidate as consolidate_univariate
from usv_playpen.modeling.consolidate_model_selection_results import consolidate as consolidate_model_selection
from usv_playpen.modeling.model_selection import (
    bout_onset_model_selection,
    vocal_category_model_selection,
)
from usv_playpen.modeling.modeling_usv_manifold_position import ContinuousModelingPipeline
from usv_playpen.modeling.modeling_vocal_bout_parameters import BoutParameterPipeline
from usv_playpen.modeling.modeling_vocal_categories_binomial import VocalCategoryModelingPipeline
from usv_playpen.modeling.modeling_vocal_categories_multinomial import MultinomialModelingPipeline
from usv_playpen.modeling.modeling_vocal_onsets import VocalOnsetModelingPipeline
from usv_playpen.modeling.jax_neural_network_cnn import NeuralContinuousCNNRunner

from usv_playpen.visualizations.modeling_plots import (
    DeepResultsVisualizer,
    plot_collinearity_audit,
    plot_feature_ranking,
    plot_model_selection_results,
    plot_multinomial_multivariate_filters,
    plot_multinomial_selection_diagnosis,
    plot_multinomial_selection_trajectory,
    plot_raw_feature_difference,
    plot_significant_filters,
    plot_significant_filters_grid,
    plot_timescale_audit,
    plot_timescale_audit_per_feature,
    plot_univariate_multinomial_performance,
    plot_vocalization_embedding_space,
)

apply_plot_style()


## 2. Extract and save modeling input data

Each of the four extraction calls below converts session-level loader output into the modeling-input pickle that the downstream univariate / model-selection runners consume. They differ only in **what gets predicted** (and therefore which event timestamps populate the timescale audit's `Y(t)` impulse trace):

| Pipeline | Predicts | `Y(t)` impulses |
|---|---|---|
| `VocalOnsetModelingPipeline` | "is this frame the start of a bout?" | bout starts |
| `BoutParameterPipeline` | per-bout duration / complexity / intensity | bout starts |
| `MultinomialModelingPipeline` | per-USV GMM category | per-USV starts |
| `ContinuousModelingPipeline` | per-USV UMAP manifold position | per-USV starts |

**Side effects (every call):**

- Writes the modeling-input pickle to `io.save_directory` (filename embeds the cohort label, e.g. `male_mute_partner`, plus a timestamp).
- Writes the paired predictor-diagnostics audit artifacts: `*_collinearity.pkl` and `*_timescales.pkl` (visualised in the next section).

The cohort label is derived from `io.session_list_file` (see `derive_experimental_condition` in `modeling_metadata.py`).


In [ ]:
### Extract and save vocal bout onset to model

VocalOnsetModelingPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data()

In [ ]:
### Extract and save vocal bout duration/complexity/intensity data to model

BoutParameterPipeline(modeling_settings_dict=None).extract_and_save_modeling_input_data()

In [ ]:
### Extract and save category prediction data for multinomial modeling

MultinomialModelingPipeline(modeling_settings_dict=None).extract_and_save_multinomial_input_data()

In [ ]:
### Extract and save USV UMAP target position data to model

ContinuousModelingPipeline(modeling_settings_dict=None).extract_and_save_continuous_data()

## 3. Predictor diagnostics: collinearity + timescale plots

Run **after** the extract step has produced the audit artifacts. The three plots share feature ordering and per-group colour so that one feature can be cross-referenced by row position and hue across all three figures.

**What each plot answers**

- `plot_timescale_audit_per_feature` — for each predictor: how long does its ACF stay above the circular-shift null (ACF horizon, left panel), and at what lag does its cross-correlation with `Y(t)` cross out of the null envelope (XC horizon, right panel)?
- `plot_timescale_audit` — cohort summary as horizontal bars, sorted by horizon length. The triangle markers on the per-feature plot **are** the bar lengths here.
- `plot_collinearity_audit` — Spearman ρ heatmap (left) and VIF horizontal bars (right). Cells whose `|ρ|` exceeds `outline_threshold` are annotated.

**Inputs**: paths to the `*_timescales.pkl` and `*_collinearity.pkl` artifacts from the previous section. The collinearity path is derived from the timescale path by string replacement.

**Settings dependencies** read from inside the audits: `diagnostics.timescale_*` (max lag, n_shuffles, shuffle range, signal floor, min run length), `model_params.gmm_component_index`, `model_params.gmm_z_score`, `gmm_params` (for the IBI threshold reference line).


In [ ]:
### Predictor diagnostics: collinearity + timescale audit plots
#
# Run these AFTER the extract-and-save step has produced the paired
# `_collinearity.pkl` and `_timescales.pkl` audit artifacts next to the
# main modeling input pickle. The three plots are intended to be read
# side-by-side: row order, per-feature group colour, and the
# single-legend convention are deliberately aligned across all three.

save_fig_bool = False
save_fig_format = 'svg'
save_fig_directory = pathlib.Path(configure_path('/mnt/falkner/Bartul/figures'))
save_fig_directory.mkdir(exist_ok=True, parents=True)


timescale_pkl = configure_path('/mnt/falkner/Bartul/modeling/modeling_boutparam_bout_durations_male_mute_partner_20260506_122135_timescales.pkl')
collinearity_pkl = timescale_pkl.replace('_timescales.pkl', '_collinearity.pkl')

# Order matters: run the per-feature plot FIRST, then the cohort
# summary. The cohort plot (`plot_timescale_audit`) is now a
# horizontal-bar summary of the triangle markers from the
# per-feature panels — one bar per feature, length = the lag at
# which that feature confidently leaves its circular-shift null
# (ACF horizon on the left, cross-correlation horizon on the
# right). It uses exactly the same marker-finding logic as the
# per-feature plot, so the per-feature plot is the ground-truth
# view that the summary aggregates over.

# (1) Per-feature small-multiples: one row per feature, two
#     columns (left: ACF + null + horizon triangle, right:
#     cross-correlation + null + horizon triangle). The marker
#     positions on this plot ARE the per-feature horizons that
#     the cohort plot (run next) will summarise.
plot_timescale_audit_per_feature(timescale_pkl,
                                 save_plot_bool=save_fig_bool,
                                 save_dir=save_fig_directory,
                                 plot_format=save_fig_format)
# To write a file instead of inline:
# plot_timescale_audit_per_feature(timescale_pkl, save_dir='/mnt/falkner/Bartul/modeling',
#                                  plot_format='png')

# (2) Cohort timescale summary: horizontal bars of per-feature
#     horizons sorted descending, with mean (solid) and median
#     (dashed) reference lines on each panel, and a bold-numeric
#     ▲ pointer marking the cohort mean on the x-axis.
plot_timescale_audit(timescale_pkl,
                     save_plot_bool=save_fig_bool,
                     save_dir=save_fig_directory,
                     plot_format=save_fig_format)
# To write a file instead of inline:
# plot_timescale_audit(timescale_pkl, save_dir='/mnt/falkner/Bartul/modeling',
#                      plot_format='png')

# (3) Collinearity diagnostic. Two-panel figure that shares the
#     timescale plots' feature ordering (self → SEI → other →
#     social/dyadic) and per-group colour palette so a feature
#     can be cross-referenced by row position and hue across all
#     three figures:
#       Left  — Spearman ρ heatmap, group-ordered with thin
#               black separators between groups; off-diagonal
#               cells whose |ρ| crosses the audit's concern /
#               exclude thresholds receive a thin / thick black
#               outline (exclude tier additionally annotated
#               with the numeric ρ).
#       Right — Variance Inflation Factor horizontal bars,
#               sorted descending, with bar fill and y-tick
#               label colour matching the heatmap's per-group
#               palette. Mean / median reference lines plus a ▲
#               pointer at the cohort mean, mirroring the
#               cohort timescale plot's convention.
plot_collinearity_audit(collinearity_pkl,
                        save_plot_bool=save_fig_bool,
                        save_dir=save_fig_directory,
                        plot_format=save_fig_format)
# To write a file instead of inline:
# plot_collinearity_audit(collinearity_pkl, save_dir='/mnt/falkner/Bartul/modeling',
#                         plot_format='png')


## 4. Model selection (forward stepwise)

Greedy forward-stepwise feature selection on top of a univariate ranking. Each iteration adds the feature whose contribution most improves the held-out score (one-SE rule applied; see the model-selection code for the exact stopping criterion).

**Two functions, two question types**

- `bout_onset_model_selection` — binary classification: does a bout occur at this frame?
- `vocal_category_model_selection` — binary classification: is this USV in `target_category` vs all others?

Both consume:

1. `univariate_results_path`: the consolidated univariate pickle (output of section 5 below, OR a hand-managed legacy file).
2. `input_data_path`: the modeling-input pickle from section 2.
3. `output_directory`: per-step pickles are written here (one per accepted feature). Section 5's `consolidate_model_selection` later merges them.

**Tunables**: `use_top_rank_as_anchor=True` seeds step 0 with the top univariate feature; `p_val=0.01` is the per-step acceptance threshold.

Heads-up: these calls run on the local node. The HPC dispatcher (`main_model_selection_dispatcher.py`) parallelises the inner loop and is the right entry point for cohort-scale runs.


In [ ]:
### Perform model selection for bout onset prediction

bout_onset_model_selection(univariate_results_path='/mnt/falkner/Bartul/modeling/gam_results_female_bout_lam=null_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                           input_data_path='/mnt/falkner/Bartul/modeling/modeling_pdata_female_20251203_161303_110sess_bout_hist4s.pkl',
                           output_directory='/mnt/falkner/Bartul/modeling/model_selection_results/bout_onset/male/intact_partner',
                           use_top_rank_as_anchor=True,
                           p_val=0.01)

In [ ]:
### Perform model selection for USV categories

vocal_category_model_selection(univariate_results_path='/mnt/falkner/Bartul/modeling/gam_results_male_category_3_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                               input_data_path='/mnt/falkner/Bartul/modeling/modeling_category_3_male_FullSyntax_20260109_120031_hist4s.pkl',
                               output_directory='/mnt/falkner/Bartul/modeling/model_selection_results/vocal_categories/male/intact_partner',
                               use_top_rank_as_anchor=True,
                               p_val=0.01)

## 5. Consolidate per-feature univariate + per-step model-selection pickles

Run this **after** the SLURM job array (`main_univariate_dispatcher.py` / `main_model_selection_dispatcher.py`) finishes. The consolidators:

- assert metadata equality across every per-feature / per-step pickle (guards against stray files from a different run);
- hoist the agreed-upon `_input_metadata` / `_run_metadata` / `_univariate_metadata` blocks to the top of the consolidated artifact;
- emit a self-describing filename built from those blocks.

Use `delete_individuals_after=True` once you have verified the consolidated artifact is correct (the per-feature files become redundant and bloat the directory). `move_to_steps_subdir=True` on the model-selection consolidator relocates the consumed step pickles into `<input_dir>/steps/` so the working directory stays tidy.


In [ ]:
### Consolidate per-feature univariate pickles + per-step model-selection pickles
#
# Run this cell *after* the SLURM job array finishes. The consolidators:
#   - assert metadata equality across every per-feature / per-step pickle
#     (guards against stray files from a different run)
#   - hoist the agreed-upon `_input_metadata` / `_run_metadata` /
#     `_univariate_metadata` blocks to the top of the consolidated artifact
#   - emit a self-describing filename built from those blocks
# Pass `delete_individuals_after=True` once you have verified the consolidated
# artifact is correct (the per-feature files are then redundant).

# 1. Univariate per-feature -> consolidated `univariate_<tag>_<condition>_<ts>.pkl`
consolidate_univariate(
    input_dir='/mnt/falkner/Bartul/modeling/cluster/univariate_results_multi_file/male/male_multinomial_vae_supercategory',
    delete_individuals_after=False,
)


In [ ]:
# 2. Model-selection per-step -> consolidated
#    `selection_<tag>_<condition>_<selection_function>_<ts>.pkl`
# `move_to_steps_subdir=True` relocates the consumed step pickles into a
# `<input_dir>/steps/` folder so the working directory stays tidy without
# losing the per-step crash-recovery files.
#
# `ignore_provenance_keys` extends the default
# `('git_commit', 'git_dirty', 'package_version')` with `'settings_sha256'`.
# Step pickles in long-running selections often span days; if
# `_parameter_settings/analyses_settings.json` is edited mid-run in a way
# that doesn't touch the selection-relevant fields (a comment, an
# unrelated key, whitespace), the file's overall hash changes and the
# consolidator's `_run_metadata` equality check would otherwise abort
# with a `settings_sha256` mismatch. Drop the override if you want the
# safety net back on.
consolidate_model_selection(
    input_dir='/mnt/falkner/Bartul/modeling/model_selection_results/male/bout_onset',
    move_to_steps_subdir=False,
    ignore_provenance_keys=(
        'git_commit', 'git_dirty', 'package_version', 'settings_sha256',
    ),
)

## 6. Univariate-model visualisations

Plots over the consolidated **univariate** pickle (one fit per behavioural feature, no forward stepping). Use these to triage which features look promising before committing to model selection or to compare cohorts.

- `plot_feature_ranking` — per-feature bar chart of the chosen primary metric, with the secondary metric overlaid. P-value-thresholded features are highlighted.
- `plot_significant_filters` — small multiples of each significant feature's learned filter (one panel per feature).
- `plot_significant_filters_grid` — same filters laid out on a single grid, baseline-corrected for direct cross-feature comparison.
- `plot_raw_feature_difference` — distribution of raw (z-scored) feature values between two conditions, with bootstrapped CIs. Quick sanity check that the feature carries any signal at all.


In [ ]:
### Plot feature ranking for univariate models (each metric gets its own plot)

plot_feature_ranking(results_file_loc='/mnt/falkner/Bartul/modeling/univariate_results/univariate_multinomial_vae_supercategory_intact_partners_male_20260511_204026Z.pkl',
                     p_val=0.01,
                     evaluation_metric='ll',
                     evaluation_metric_name='Negative Log-Likelihood',
                     secondary_metric='score',
                     secondary_metric_name="Accuracy",
                     ignore_features=None,
                     save_plot=False,
                     output_dir='/mnt/falkner/Bartul/modeling/figures')


In [ ]:
### Plot feature ranking for univariate models for multinomial category

plot_univariate_multinomial_performance(
      results_file_loc='/mnt/falkner/Bartul/modeling/univariate_results/univariate_multinomial_vae_supercategory_intact_partners_male_20260511_204026Z.pkl',
      evaluation_metric='ll',
      evaluation_metric_name='Negative Log-Likelihood',
      secondary_metric='score',
      secondary_metric_name='Balanced Accuracy',
      p_val_threshold=0.01,
      save_plot=False,
      output_dir='/mnt/falkner/Bartul/modeling/figures',
  )

In [ ]:
### Plot individual filters of univariate models

plot_significant_filters(results_file_loc='/mnt/falkner/Bartul/modeling/gam_results_male_mute_partner_category_18_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                         metric='ll',
                         ignore_features=None,
                         p_val=0.01,
                         save_plot=False,
                         output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot all significant filters in a grid, baseline-corrected, considering univariate models

plot_significant_filters_grid(results_file_loc='/mnt/falkner/Bartul/modeling/gam_results_male_category_10_lam=0.6_max_iter=500_n_splines_time=8_n_splines_value=5.pkl',
                              ignore_features=None,
                              metric='ll',
                              p_val_threshold=0.01,
                              save_plot=False,
                              output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot raw (z-scored) feature differences

plot_raw_feature_difference(pickle_file_path='/mnt/falkner/Bartul/modeling/modeling_category_3_male_presence_all_20260129_193701_hist4s.pkl',
                            feature_key='self.neck_elevation',
                            feature_color='#9AC0CD',
                            subset_fraction=0.05,
                            n_bootstraps=100,
                            save_plots=False,
                            output_dir='/mnt/falkner/Bartul/modeling/figures')

## 7. Model-selection (forward-stepwise) visualisations

Plot of the per-step performance trajectory written by `bout_onset_model_selection` / `vocal_category_model_selection` (section 4). One line per metric; the X axis is the iteration number (= number of features in the model at that step).


In [ ]:
### Plot model selection results

plot_model_selection_results(selection_results_path='/mnt/falkner/Bartul/modeling/model_selection_results/bout_params/bout_onset/female',
                             metric_secondary='auc',
                             save_plots=False,
                             output_dir='/mnt/falkner/Bartul/modeling/figures')


## 8. Multinomial-specific visualisations

These four plots assume a **multinomial** model (one-vs-rest classifier across all GMM categories). They read either the consolidated univariate multinomial pickle or the consolidated multinomial selection-results directory.

- `plot_univariate_multinomial_performance` — per-feature, per-class metric heatmap. The `diff_cmap` panel shows performance relative to the cohort mean.
- `plot_multinomial_selection_trajectory` — primary + secondary metric per forward-stepwise iteration, broken down by class.
- `plot_multinomial_multivariate_filters` — final selected filters (one panel per class × selected feature) with a shared diverging colormap.
- `plot_multinomial_selection_diagnosis` — how much does the multivariate selection differ from picking the top univariate feature for each class? Two heatmaps, base + difference.


In [ ]:
### Plot performance of univariate multinomial models

plot_univariate_multinomial_performance(
        results_file_loc='/mnt/scratch/Bartul/modeling/univariate_results/multinomial_categories/univariate_multinomial_categories_male_bin_resize=10_lambda=1E0_l2=1E-1.pkl',
        evaluation_metric='auc',
        evaluation_metric_name='Area Under the ROC Curve',
        secondary_metric='score',
        secondary_metric_name='Balanced Accuracy',
        p_val_threshold=0.05,
        diff_cmap='bwr',
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot performance trajectory of model selection for multinomial models

plot_multinomial_selection_trajectory(
        selection_results_path='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner',
        metric_primary='auc',
        primary_metric_name='Area Under the ROC Curve',
        metric_secondary='score',
        secondary_metric_name="Balanced Accuracy",
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures')

In [ ]:
### Plot filters of multinomial models (final model selection)

plot_multinomial_multivariate_filters(
        selection_results_path='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male',
        history_window_sec=4.0,
        cmap='bwr',
        save_plot=True,
        output_dir='/mnt/falkner/Bartul/modeling/figures'
)

In [ ]:
### Plot how much te final model selection for multinomial models differs from the top-ranked univariate model

plot_multinomial_selection_diagnosis(
        selection_results_path='/mnt/scratch/Bartul/modeling/model_selection_results/multinomial_category/male_mute_partner',
        cmap_diff='bwr',
        save_plot=True,
        output_dir='/mnt/falkner/Bartul/modeling/figures'
)

## 9. CNN deep-learning pipeline (continuous manifold position)

Non-linear baseline for the continuous manifold-position regression. Three cells:

1. **Load** — `runner.load_multivariate_data_blocks(...)` reads the modeling-input pickle from section 2 and stacks the per-feature `(N, T)` matrices into the `(N, F, T)` tensor the 1-D ResNet consumes.
2. **Train** — `runner.run_cnn_training(data_blocks=...)` runs the full training loop and writes `cnn_*_predictions_*.pkl` next to the input pickle.
3. **Visualise** — `DeepResultsVisualizer` exposes five diagnostic modes; pick one via the `choose_analysis` switch (`permutation_test`, `feature_importance`, `spatial_precision_grid`, `error_landscape`, `regional_saliency`).


In [ ]:
### Run CNN to predict vocal manifold positions
#
# Loads the multivariate feature blocks produced by
# `ContinuousModelingPipeline.extract_and_save_continuous_data()` and
# constructs the 3-D tensor the 1-D ResNet expects. `data_blocks` is
# then handed to `runner.run_cnn_training(...)` in the next cell.

runner = NeuralContinuousCNNRunner(modeling_settings=None)

data_blocks = runner.load_multivariate_data_blocks(pkl_path='/mnt/falkner/Bartul/modeling/modeling_UMAP_manifold_position_male_20260226_145551_hist4s.pkl')


In [ ]:
runner.run_cnn_training(data_blocks=data_blocks)

In [ ]:
### Plot CNN results in predicting vocal manifold positions
#
# Choose one of the five visualisation modes via `choose_analysis`.
# Each mode reads the same `cnn_*_predictions_*.pkl` artifact written
# at the end of `runner.run_cnn_training(...)` and renders a different
# diagnostic projection of the trained network's behaviour on the
# held-out folds.

deep_visualizer = DeepResultsVisualizer(results_pkl_path='/mnt/falkner/Bartul/modeling/neural_network_results/cnn_manifold_integrated_predictions_male_20260326_220132.pkl',
                                        modeling_settings=None,
                                        visualization_settings=None)

choose_analysis = 'regional_saliency'  # 'error_landscape' 'spatial_precision_grid' 'feature_importance' 'permutation_test'

if choose_analysis == 'permutation_test':
    deep_visualizer.plot_permutation_test(save_plot=False,
                                          output_dir='/mnt/falkner/Bartul/modeling/figures',
                                          file_format='png')
elif choose_analysis == 'feature_importance':
    deep_visualizer.plot_feature_importance(snr_threshold=3.0,
                                            error_bar_color='#000000',
                                            save_plot=False,
                                            output_dir='/mnt/falkner/Bartul/modeling/figures',
                                            file_format='png')
elif choose_analysis == 'spatial_precision_grid':
    deep_visualizer.plot_spatial_precision_grid(n_patches=20,
                                                min_samples=25,
                                                plot_type='density',
                                                bg_pt_color='#E0E0E0',
                                                peak_pt_color='cyan',
                                                square_edge_color='#000000',
                                                panel_fontsize=9,
                                                figsize_unit=3.0,
                                                save_plot=False,
                                                output_dir='/mnt/falkner/Bartul/modeling/figures',
                                                file_format='png')
elif choose_analysis == 'error_landscape':
    deep_visualizer.plot_error_landscape(gridsize=30,
                                         vmax_percentile=95.0,
                                         save_plot=False,
                                         output_dir='/mnt/falkner/Bartul/modeling/figures',
                                         file_format='png')
elif choose_analysis == 'regional_saliency':
    deep_visualizer.plot_regional_saliency_inset(
        source_data_path='',
        region_key='categories_1_and_2',
        category_name='Complex USVs',
        prediction_plot_type='density',
        save_plot=False,
        output_dir='/mnt/falkner/Bartul/modeling/figures',
        file_format='png')
else:
    print(f"Option {choose_analysis} not recognized.")

## 10. Vocalisation embedding-space view

Renders the UMAP-embedded USV space with cluster-category overlays and a grid of per-cell mean spectrograms. This is a corpus-level descriptive plot rather than a model output — useful for orienting yourself to the structure of the GMM categories before reading the modelling figures above.

**Inputs**

- `umap_position_file_path` — H5 with the per-session UMAP coordinates.
- `cluster_category_file_path` — H5 with the per-session cluster-category dictionary.
- `target_subdirs` — limit the corpus to a list of dataset roots.


In [ ]:

plot_vocalization_embedding_space(umap_position_file_path='/mnt/falkner/Charlie/USV_town/h5_files_ch/whole_dset_plus_ephys/session_to_umap_dict.h5',
                                  cluster_category_file_path='/mnt/falkner/Charlie/USV_town/h5_files_ch/whole_dset_plus_ephys/clustering/cluster_dict_6clus.h5',
                                  target_subdirs=["Liza/data", "Jinrun/Data", "Bartul/Data"],
                                  csv_category_column_id='usv_supercategory',
                                  x_range=(15.0, 17.5),
                                  y_range=(4.5, 6.5),
                                  grid_dims=(4, 4),
                                  save_plot=False,
                                  output_dir='/mnt/falkner/Bartul/modeling/figures')